# Attention/UQ benchmark — standalone 800-question Colab

This notebook evaluates **Qwen/Qwen2-VL-7B-Instruct in unquantized FP16** on a fixed 200-question subset from each of:

1. MultiModalQA (raw images, tables, and text)
2. WebQA (full paired image plus a documented constructed distractor set)
3. HotpotQA distractor validation (text)
4. TAT-QA dev (tables and text)

It requires no manual file upload. The compact, deterministic 800-question bundle is downloaded from Hugging Face. Images are passed as actual image files; Qwen's standard processor may dynamically resize them while preserving the complete frame.

The attention implementation uses a cached prefix followed by a one-token attention pass. This recovers the final prompt token's attention without retaining every layer's full quadratic prefill matrix, which was the cause of the earlier 65–76 GiB OOMs.


In [ ]:
# Standalone Colab bootstrap — all Python packages are installed with uv.
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/paper")
BUNDLE_DIR = Path("/content/attention_uq_800q")
CODE_REPO = "https://github.com/NguyenKhanh2603/Uncertainty-Aware-Iterative-RAG.git"
CODE_COMMIT = "169010664834d4a3699e1908054b08e3839c0714"
DATA_REPO = "danny2507/attention-uq-800q-colab"
DATA_REVISION = "3dd72ff2ddb2cd2cc0937b35018d836707d92701"

packages = [
    "transformers==4.46.3",
    "accelerate>=0.34.2",
    "qwen-vl-utils==0.0.8",
    "huggingface_hub>=0.26.0",
    "openai>=1.30.0",
    "pydantic>=2.5.0",
    "pyyaml>=6.0",
    "Pillow>=10.0.0",
    "numpy>=1.26.0",
    "pandas>=2.0.0",
    "scipy>=1.12.0",
    "scikit-learn>=1.4.0",
    "tqdm>=4.66.0",
]

uv_executable = shutil.which("uv")
if uv_executable is None:
    installer = Path("/tmp/install-uv.sh")
    subprocess.check_call(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", str(installer)])
    subprocess.check_call(["sh", str(installer)])
    uv_executable = str(Path.home() / ".local" / "bin" / "uv")
if not Path(uv_executable).is_file():
    raise RuntimeError(f"uv installation failed: {uv_executable}")
subprocess.check_call([uv_executable, "pip", "install", "--system", *packages])

# Colab may retain /content/paper after a previously interrupted clone.
# Remove only that incomplete checkout; a valid Git checkout is reused.
if PROJECT_DIR.exists() and not (PROJECT_DIR / ".git").is_dir():
    print(f"Removing incomplete checkout: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

# Accept a Markdown-wrapped URL too, in case it was pasted from rendered text.
if CODE_REPO.startswith("[") and "](" in CODE_REPO and CODE_REPO.endswith(")"):
    CODE_REPO = CODE_REPO.split("](", 1)[1][:-1]
if not CODE_REPO.startswith(("https://", "http://")):
    raise ValueError(f"Invalid CODE_REPO URL: {CODE_REPO!r}")

if not (PROJECT_DIR / ".git").is_dir():
    subprocess.check_call(["git", "clone", "--filter=blob:none", CODE_REPO, str(PROJECT_DIR)])
subprocess.check_call(["git", "-C", str(PROJECT_DIR), "fetch", "origin", CODE_COMMIT])
subprocess.check_call(["git", "-C", str(PROJECT_DIR), "checkout", "--detach", CODE_COMMIT])

# The pinned client defaults to BF16 on capable GPUs. This benchmark explicitly
# keeps the user's requested unquantized FP16 weights.
client_file = PROJECT_DIR / "src" / "uncertainty_rag" / "models" / "llm_client.py"
source = client_file.read_text(encoding="utf-8")
default_line = 'model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16'
fp16_line = 'model_kwargs["torch_dtype"] = torch.float16'
if default_line in source:
    client_file.write_text(source.replace(default_line, fp16_line, 1), encoding="utf-8")
elif fp16_line not in source:
    raise RuntimeError("Could not enforce FP16 in HuggingFaceLocalClient")

os.chdir(PROJECT_DIR)
for path in (PROJECT_DIR, PROJECT_DIR / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from huggingface_hub import snapshot_download
snapshot_download(
    repo_id=DATA_REPO,
    repo_type="dataset",
    revision=DATA_REVISION,
    local_dir=BUNDLE_DIR,
    allow_patterns=[
        "manifest.json",
        "attention_uq_800q_runner.py",
        "mmqa/**",
        "webqa/**",
        "hotpotqa/**",
        "tatqa/**",
    ],
)

manifest = __import__("json").loads((BUNDLE_DIR / "manifest.json").read_text(encoding="utf-8"))
if manifest.get("total_questions") != 800:
    raise RuntimeError(f"Expected 800 bundled questions, got {manifest.get('total_questions')}")
for dataset in ("mmqa", "webqa", "hotpotqa", "tatqa"):
    path = BUNDLE_DIR / dataset / "questions.jsonl"
    count = sum(1 for _ in path.open(encoding="utf-8"))
    if count != 200:
        raise RuntimeError(f"Expected 200 {dataset} questions, got {count}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before running this notebook")
gpu = torch.cuda.get_device_properties(0)
print(f"Ready: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB), 4 datasets × 200 questions")
print("Bundle:", DATA_REPO)
print("Pinned data revision:", DATA_REVISION)


## Run and download

The runner checkpoints every completed question into `/content/attention_uq_800q_results`. If an individual method still OOMs, its earlier baseline results remain saved and the benchmark continues. At the end, Colab downloads one ZIP containing all per-dataset CSVs, the combined 800-row CSV, summary JSON, and OOM log.

Expected runtime is substantially longer than the 100-question pilot. Keep the browser/runtime connected until the ZIP download begins.


In [ ]:
# Optional overrides must be set before this cell, for example:
# os.environ["ATTN_UQ_N_PER_DATASET"] = "5"   # smoke test
# os.environ["ATTN_UQ_WINDOW_SIZE"] = "10"

runner = BUNDLE_DIR / "attention_uq_800q_runner.py"
if not runner.is_file():
    raise FileNotFoundError(f"Missing bundled runner: {runner}")
exec(compile(runner.read_text(encoding="utf-8"), str(runner), "exec"), globals())
